# Lab 04 — Feature Extraction, Filtering & Edge Detection
**There is no separate Lab 04 task sheet.** The manual contains one explicit task,
*"Texture Analysis for Material Classification (Give Code and Description)"*, which is
the first task cell. The remaining cells (E1–E10) implement each technique the manual
demonstrates, so every Lab 4 topic has a ready, runnable cell.
Run **Setup** first; set `IMAGE_PATH` to your own image (synthetic fallback otherwise).

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import hog, local_binary_pattern
from skimage import exposure

IMAGE_PATH = "image.jpg"
os.makedirs("output", exist_ok=True)


def show(imgs, titles, cols=4, cmap="gray"):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(4.5 * cols, 4 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap=cmap)
        plt.title(t, fontsize=10); plt.axis("off")
    plt.tight_layout(); plt.show()


def synthetic_scene():
    img = np.full((360, 480, 3), (210, 200, 190), np.uint8)
    cv2.rectangle(img, (40, 40), (200, 200), (40, 80, 200), -1)
    cv2.circle(img, (330, 130), 80, (60, 170, 60), -1)
    pts = np.array([[80, 330], [240, 230], [420, 330]], np.int32)
    cv2.fillPoly(img, [pts], (180, 60, 40))
    for x in range(250, 460, 14):
        cv2.line(img, (x, 220), (x, 260), (20, 20, 20), 3)
    cv2.putText(img, "LAB 4", (40, 300), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 0), 3)
    return img


img = cv2.imread(IMAGE_PATH)
if img is None:
    print(f"'{IMAGE_PATH}' not found -> synthetic scene")
    img = synthetic_scene()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Setup done | image", img.shape)

In [ ]:
# ===================== TASK — Texture Analysis for Material Classification =====================
# Classify wood, metal and fabric from texture features. Two techniques from the manual:
#   (1) LBP histogram   (2) texture energy & contrast (local statistics)
# Patches are cut from one texture image per material (replace with real photos via MATERIAL_PATHS),
# half are used to "train" (class mean feature), half to test with nearest-centroid.
MATERIAL_PATHS = {"wood": "wood.jpg", "metal": "metal.jpg", "fabric": "fabric.jpg"}
rng = np.random.default_rng(0)


def synth_material(name, size=384):
    y, x = np.mgrid[0:size, 0:size].astype(np.float64)
    if name == "wood":       # wavy growth rings
        g = 128 + 70 * np.sin(0.12 * np.sqrt((x - 40) ** 2 + (y * 3 - 300) ** 2) + 2 * np.sin(x / 40))
    elif name == "metal":    # brushed: fine horizontal streaks
        g = 150 + cv2.GaussianBlur(rng.normal(0, 60, (size, size)), (61, 1), 0) * 3
    else:                    # fabric: woven over/under pattern
        g = 128 + 60 * np.sign(np.sin(x / 3)) * np.sign(np.sin(y / 3)) + 30 * np.sin(x / 1.5)
    g += rng.normal(0, 8, g.shape)
    return np.clip(g, 0, 255).astype(np.uint8)


materials = {}
for name, p in MATERIAL_PATHS.items():
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    materials[name] = im if im is not None else synth_material(name)


def lbp_features(patch, P=8, R=1):
    lbp = local_binary_pattern(patch, P, R, method="uniform")      # rotation-invariant uniform codes
    h, _ = np.histogram(lbp, bins=np.arange(P + 3))                # P+2 bins
    return h / h.sum()


def energy_contrast_features(patch, n=7):
    f = patch.astype(np.float32)                                   # float: uint8**2 overflows
    energy = cv2.blur(f ** 2, (n, n))                              # local mean of I^2
    contrast = np.sqrt(np.maximum(energy - cv2.blur(f, (n, n)) ** 2, 0))   # local std
    return np.array([energy.mean() / 255 ** 2, energy.std() / 255 ** 2,
                     contrast.mean() / 128, contrast.std() / 128])


def features(patch):
    return np.concatenate([lbp_features(patch), energy_contrast_features(patch)])


X, y = [], []
for name, im in materials.items():
    for r in range(0, im.shape[0] - 63, 64):
        for c in range(0, im.shape[1] - 63, 64):
            X.append(features(im[r:r + 64, c:c + 64])); y.append(name)
X, y = np.array(X), np.array(y)
idx = rng.permutation(len(y)); half = len(y) // 2
tr, te = idx[:half], idx[half:]
centroids = {k: X[tr][y[tr] == k].mean(axis=0) for k in materials}
pred = np.array([min(centroids, key=lambda k: np.linalg.norm(x - centroids[k])) for x in X[te]])
acc = (pred == y[te]).mean()
print(f"Patches: {len(y)} | train {len(tr)} / test {len(te)} | nearest-centroid accuracy = {acc:.2%}")
print(pd.crosstab(pd.Series(y[te], name="true"), pd.Series(pred, name="pred")))

plt.figure(figsize=(15, 7))
for i, (name, im) in enumerate(materials.items()):
    lbp = local_binary_pattern(im, 8, 1, method="uniform")
    plt.subplot(2, 3, i + 1); plt.imshow(im, cmap="gray"); plt.title(name); plt.axis("off")
    plt.subplot(2, 3, i + 4); plt.bar(range(10), lbp_features(im)); plt.title(f"{name}: LBP histogram")
plt.tight_layout(); plt.show()

print("""
DESCRIPTION
1) Local Binary Patterns (LBP): each pixel's P neighbours on a circle of radius R are compared
   with the centre (>= centre -> 1, else 0), giving a P-bit code; 'uniform' codes (<= 2 bit
   transitions) are grouped into P+2 bins and the histogram of codes is the texture descriptor.
   Suits fine micro-textures: fabric weave, grain, rough vs smooth surfaces.
2) Texture energy & contrast: in an n x n window, energy = mean of I^2 (overall strength)
   and contrast = local standard deviation (roughness). Suits coarse differences: smooth
   polished metal (low contrast) vs rough wood/fabric (high contrast).

ADVANTAGES / LIMITATIONS
- LBP: + cheap, illumination-robust (only relative comparisons), rotation-invariant with
  'uniform'. - noise-sensitive (single-pixel comparisons), fixed scale (R), discards
  contrast magnitude, so two textures with the same pattern but different strength look alike.
- Energy/contrast: + very simple, captures roughness and brightness. - depends on lighting
  and camera exposure, no directional information (brushed metal vs fabric can share stats),
  window size must match the texture scale.
- Combining both (as above) covers each other's blind spots; real material datasets also need
  scale/lighting variation in training.""")

In [ ]:
# ===================== E1 — HOG (Histogram of Oriented Gradients) =====================
small = cv2.resize(gray, (128, 128))
feat, hog_img = hog(small, orientations=9, pixels_per_cell=(8, 8),
                    cells_per_block=(2, 2), visualize=True)
hog_img = exposure.rescale_intensity(hog_img, in_range=(0, 10))
print("HOG feature vector length:", feat.shape[0], "(15x15 blocks x 2x2 cells x 9 bins)")
show([small, hog_img], ["Input 128x128", "HOG visualisation"], cols=2)

In [ ]:
# ===================== E2 — LBP (Local Binary Patterns) =====================
P, R = 8, 1
lbp = local_binary_pattern(gray, P, R, method="uniform")
hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, P + 3), range=(0, P + 2))
hist = hist / hist.sum()
print("LBP histogram (normalised, P+2 bins):", np.round(hist, 3))
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(lbp, cmap="gray"); plt.title("LBP image"); plt.axis("off")
plt.subplot(1, 2, 2); plt.bar(range(len(hist)), hist); plt.title("LBP histogram"); plt.show()

In [ ]:
# ===================== E3 — Colour histogram =====================
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for i, col in enumerate("bgr"):
    plt.plot(cv2.calcHist([img], [i], None, [256], [0, 256]), color=col)
plt.title("BGR histograms"); plt.xlim(0, 256)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
hs = cv2.calcHist([hsv], [0, 1], None, [8, 8], [0, 180, 0, 256])      # H: 0-180, S: 0-256
hs = cv2.normalize(hs, hs).flatten()
plt.subplot(1, 2, 2); plt.bar(range(64), hs); plt.title("HSV (H x S, 8x8) feature vector")
plt.show()
print("Colour feature length:", hs.shape[0])

In [ ]:
# ===================== E4 — HED (Histogram of Edge Directions) =====================
blur = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blur, 30, 70)
gx = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
direction = np.rad2deg(np.arctan2(gy, gx)) % 360
hed, bins = np.histogram(direction[edges > 0], bins=36, range=(0, 360))   # edge pixels only
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(edges, cmap="gray"); plt.title("Canny edges"); plt.axis("off")
plt.subplot(1, 2, 2); plt.bar(bins[:-1], hed, width=10); plt.title("Histogram of Edge Directions")
plt.xlabel("degrees"); plt.show()
print("HED feature vector:", hed)

In [ ]:
# ===================== E5 — HIG (Histogram of Intensity Gradients) =====================
mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
hig, bins = np.histogram(ang, bins=36, range=(0, 360), weights=mag)       # all pixels, magnitude-weighted
plt.figure(figsize=(6, 4)); plt.bar(bins[:-1], hig, width=10)
plt.title("Histogram of Intensity Gradients (magnitude-weighted)"); plt.xlabel("degrees"); plt.show()
print("HED uses only edge pixels (counts); HIG uses every pixel weighted by gradient magnitude.")

In [ ]:
# ===================== E6 — Texture energy & contrast =====================
n = 5
f = gray.astype(np.float32)
energy = cv2.filter2D(f ** 2, -1, np.ones((n, n), np.float32) / (n * n))
mean = cv2.blur(f, (n, n))
contrast = np.sqrt(np.maximum(cv2.blur(f ** 2, (n, n)) - mean ** 2, 0))
e_hist, e_bins = np.histogram(energy, bins=256, range=(0, energy.max()))
c_hist, c_bins = np.histogram(contrast, bins=256, range=(0, contrast.max()))
show([gray, energy, contrast], ["Gray", f"Energy ({n}x{n})", f"Contrast ({n}x{n})"], cols=3)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.plot(e_bins[:-1], e_hist / e_hist.sum(), "b"); plt.title("Texture Energy Histogram")
plt.subplot(1, 2, 2); plt.plot(c_bins[:-1], c_hist / c_hist.sum(), "r"); plt.title("Texture Contrast Histogram")
plt.tight_layout(); plt.show()

In [ ]:
# ===================== E7 — Filtering: smoothing, sharpening, emboss =====================
box = cv2.filter2D(img, -1, np.ones((5, 5), np.float32) / 25)
g1 = cv2.getGaussianKernel(5, 1.0)                    # 5x1 column -> outer product = 5x5
gauss = cv2.filter2D(img, -1, g1 @ g1.T)
median = cv2.medianBlur(img, 5)
bilateral = cv2.bilateralFilter(img, 9, 75, 75)
sharp = cv2.filter2D(img, -1, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32))
emboss = cv2.filter2D(gray, -1, np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], np.float32))
show([img, box, gauss, median, bilateral, sharp, emboss],
     ["Original", "Box 5x5", "Gaussian 5x5", "Median 5", "Bilateral", "Sharpen", "Emboss"])

In [ ]:
# ===================== E8 — Convolution types: same, valid, strided =====================
# NOTE: cv2.filter2D has NO 'strides' argument (the manual's code errors). Slice instead.
k = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], np.float32)
same = cv2.filter2D(gray, cv2.CV_32F, k, borderType=cv2.BORDER_CONSTANT)  # zero padding, same size
valid = same[1:-1, 1:-1]                                                  # no padding: lose K//2 border
strided = valid[::2, ::2]                                                 # stride 2
true_conv = cv2.filter2D(gray, cv2.CV_32F, cv2.flip(k, -1))               # real convolution = flipped kernel
H, W = gray.shape
out = lambda N, K, P, S: (N - K + 2 * P) // S + 1
print(f"input {gray.shape} | same {same.shape} | valid {valid.shape} (formula {out(H, 3, 0, 1)}x{out(W, 3, 0, 1)}) "
      f"| stride 2 {strided.shape} (formula {out(H, 3, 0, 2)}x{out(W, 3, 0, 2)})")
show([cv2.convertScaleAbs(same), cv2.convertScaleAbs(valid), cv2.convertScaleAbs(strided),
      cv2.convertScaleAbs(true_conv)], ["Same (zero pad)", "Valid", "Valid, stride 2", "True convolution"])

In [ ]:
# ===================== E9 — Sobel gradients: magnitude, direction, threshold =====================
gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)       # CV_64F keeps negative gradients
gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
magnitude = np.sqrt(gx ** 2 + gy ** 2)
direction = np.arctan2(gy, gx)
edges = (magnitude > 100).astype(np.uint8) * 255
show([gray, cv2.convertScaleAbs(gx), cv2.convertScaleAbs(gy), cv2.convertScaleAbs(magnitude),
      direction, edges], ["Gray", "Sobel x (vertical edges)", "Sobel y (horizontal edges)",
                          "Magnitude", "Direction (rad)", "Magnitude > 100"], cols=3)

In [ ]:
# ===================== E10 — Canny edge detector =====================
blur = cv2.GaussianBlur(gray, (5, 5), 1.4)            # step 1: smoothing
results, titles = [gray], ["Gray"]
for lo, hi in [(50, 100), (100, 200), (150, 250)]:
    e = cv2.Canny(blur, lo, hi)                        # steps 2-4: gradient, NMS, hysteresis
    results.append(e); titles.append(f"Canny {lo}/{hi}: {np.count_nonzero(e)} px")
v = np.median(gray)
auto = cv2.Canny(blur, int(max(0, 0.67 * v)), int(min(255, 1.33 * v)))
results.append(auto); titles.append("Auto (median-based)")
show(results, titles, cols=5)
cv2.imwrite("output/e10_canny.png", results[2])